In [15]:
from nltk.corpus import movie_reviews

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import re

In [17]:
# 사용자 정의 토크나이져
class SimpleTokenizer:
    def __init__(self,num_words = 10000, oov_token='UNK'):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {}
        self.index_word = {}

    def _clean_text(self,text):
        return re.findall(r'\w+',text.lower())  # re.sub(r'[^\w\s]','',temp).strip()
    
    def fit_on_texts(self,texts):
        '''빈도순으로 상위 단어 추출 토큰을 숫자로 변경 공백을 기준으로 토큰분류'''
        word_counts = Counter()
        for text in texts:
            word_counts.update(self._clean_text(text))
        # 빈도순으로 num_words 단어 추출
        most_common = word_counts.most_common(self.num_words-2)  # pad unk 특수토큰 자리 남겨
        # 0:padding 1: oov(out of vocaburay)
        self.word_index = {self.oov_token : 1}
        for i, (word,_) in enumerate(most_common):
            self.word_index[word] = i + 2
        self.index_word = {idx : w for w, idx in self.word_index.items()}
    def texts_to_sequences(self, texts):
        sequence = []
        for text in texts:
            seq = []
            for word in self._clean_text(text):
                seq.append(self.word_index.get(word,1))
            sequence.append(seq)
        return sequence

def pad_sequence(seqeuces, maxlen ,padding='pre', truncating='pre'):
    features = np.zeros((len(seqeuces), maxlen), dtype=int)
    for i,seq in enumerate(seqeuces):
        if len(seq) > maxlen:
            if truncating == 'pre':
                features[i, :] = seq[-maxlen:]
            else:
                features[i, :] = seq[:maxlen]
        else:
            if padding == 'pre':
                features[i, -len(seq):] = seq
            else:
                features[i, :len(seq)] = seq
    return features

In [18]:
# 리뷰데이터로 적용해서 오류없는지 확인 및 수정
reviews = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]

In [19]:
texts = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(texts) # 문자 -> 숫자
requens = sim.texts_to_sequences(texts) # 길이를 맞춤

In [20]:
features = pad_sequence(requens,500)

In [21]:
features.shape

(2, 500)

In [22]:
# RNN 모델
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim)
        self.rnn = nn.RNN(embedding_dim,hidden_dim,batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim,32),
            nn.ReLU(),
            nn.Linear(32,1),            
        )
    def forward(self, x):
        x =self.embedding(x)
        _,hn = self.rnn(x)  # output, hn 
        # output : 모든시점(time-step) 의 숨겨진 상태  : 각 단어(시점)를 거칠때마다 계산된 모든 hidden state를 모아놓음
        # 시점 1~N
        # seq2seq 모델 은 각 단어마다 결과를 내야하는 개체명 인식(NER)

        # hn : 마지막 시점의 상태 : 전체문장을 다 읽고 최종적으로 요약한 정보 : 문서분류 
        return self.fc(hn.squeeze(0))


In [23]:
# 테스트
texts = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(texts) # 문자 -> 숫자
requens = sim.texts_to_sequences(texts) # 길이를 맞춤
features = pad_sequence(requens,500)
features = torch.LongTensor(features)
print(features.shape)
features = nn.Embedding(500,32)(features)
outputs, hn = nn.RNN(32,64,batch_first=True)(features)
outputs.shape, hn.shape

torch.Size([2, 500])


(torch.Size([2, 500, 64]), torch.Size([1, 2, 64]))

In [24]:
# 데이터를 가져오기
# x y 분할
# 토크나이져 + pad_sequence   --> 문자를 숫자로 변환

# train test split
# TorchTensor로 변환
# TensorDataset - > Dataloader

# 모델생성
# 옵티마이져 정의
# 손실함수 정의

In [25]:
fileids = movie_reviews.fileids()
reviews = [movie_reviews.raw(fileid) for fileid in fileids]
categories = [movie_reviews.categories(fileid)[0] for fileid in fileids]

max_words = 10000
maxlen = 500
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = SimpleTokenizer(num_words=max_words)
tokenizer.fit_on_texts(reviews)
X = tokenizer.texts_to_sequences(reviews)
X = pad_sequence(X, maxlen=maxlen)

label_dict = {'pos': 1, 'neg': 0}
y = np.array([label_dict[c] for c in categories])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

X_train_t = torch.LongTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.LongTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print('Preprocessing complete. X_train shape:', X_train.shape)

Preprocessing complete. X_train shape: (1600, 500)


In [26]:
def train_model(model, loader, optimizer, criterion, epochs=10):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += ((outputs > 0.5).float() == y_batch).sum().item()
        print(f'Epoch {epoch+1}: Loss {total_loss/len(loader):.4f}, Acc {correct/len(loader.dataset):.4f}')

model_dense = RNNModel(max_words, 32, maxlen).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer_dense = optim.RMSprop(model_dense.parameters(), lr=0.001)
train_model(model_dense, train_loader, optimizer_dense, criterion)

Epoch 1: Loss 0.7834, Acc 0.5144
Epoch 2: Loss 0.7033, Acc 0.5081
Epoch 3: Loss 0.7050, Acc 0.5194
Epoch 4: Loss 0.7080, Acc 0.4981
Epoch 5: Loss 0.7109, Acc 0.4988
Epoch 6: Loss 0.7008, Acc 0.5081
Epoch 7: Loss 0.6995, Acc 0.5119
Epoch 8: Loss 0.7042, Acc 0.4975
Epoch 9: Loss 0.6982, Acc 0.5062
Epoch 10: Loss 0.6954, Acc 0.5138


In [ ]:
# 평가
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze()            
            outputs = torch.sigmoid(outputs)            
            correct += ((outputs > 0.5).float() == y_batch).sum().item()
    return correct / len(loader.dataset)

print(f'Final Test Accuracy (model_dense): {evaluate(model_dense, test_loader):.4f}')

### 장기 의존성  (Long-Term Dependency)  
- 문장이 길어지면 앞부분의 정보를 잊어버린다

# 1. 왜 LSTM이 필요한가요? (RNN의 한계)

일반적인 RNN은 정보를 전달할 때 **'단기 기억'** 에는 강하지만, 문장이 길어지면 앞부분의 정보를 잊어버리는 **장기 의존성(Long-Term Dependency)** 문제를 가지고 있습니다.

### 🚩 문제점: 기울기 소실 (Vanishing Gradient)
문장이 길어질수록 과거의 정보가 현재로 넘어오면서 희석되거나 사라집니다. 예를 들어:
> "나는 어릴 때 **프랑스** 에서 살았어. ... (중략) ... 그래서 나는 **[ ]** 어를 할 줄 알아."

여기서 `[ ]`에 들어갈 말이 '프랑스어'임을 맞추려면 아주 먼 과거의 '프랑스'라는 정보를 기억해야 합니다. 일반 RNN은 이 정보를 끝까지 가져가지 못하고 잊어버리곤 합니다.

## 2. LSTM의 핵심 아이디어: "기억의 통로"

LSTM의 핵심은 **셀 상태(Cell State)** 라고 불리는 '컨베이어 벨트'입니다. 이 벨트는 정보를 큰 변화 없이 그대로 전달하는 역할을 하며, **게이트(Gate)** 라는 장치들이 정보를 더하거나 뺄지를 결정합니다.